In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = os.getenv("CLAUDE_MODEL")

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [12]:
import json

# For step 2, where we create an evaluation dataset. We ask Claude to do it for us.
def generate_dataset():
    prompt = """
		Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
		that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
		each representing a task that requires Python, JSON, or a Regex to complete.

		Example output:
		```json
		[
			{
				"task": "Description of task",
                "format": "json" or "python" or "regex"
                "solution_criteria" : "The required criteria for evaluating the solution."
			},
			...additional
		]
		```

		* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
		* Focus on tasks that do not require writing much code

		Please generate 3 objects.
	"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [13]:
dataset = generate_dataset()

with open("dataset.json", "w") as file:
  json.dump(dataset, file, indent=2)

dataset

[{'task': 'Parse an AWS S3 bucket name from a full S3 URI (e.g., s3://my-bucket-name/path/to/object) and extract just the bucket name',
  'format': 'regex',
  'solution_criteria': 'The regex must correctly extract the bucket name from S3 URIs in the format s3://bucket-name/... and handle bucket names with hyphens and numbers. It should return only the bucket name without the s3:// prefix or path.'},
 {'task': 'Write a Python function that takes an AWS CloudWatch log event and extracts the timestamp and message fields, returning them as a dictionary',
  'format': 'python',
  'solution_criteria': "The function must accept a CloudWatch log event dictionary, extract 'timestamp' and 'message' fields, handle cases where fields may be missing, and return a new dictionary with these two fields."},
 {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'my-data-bucket'",
  'format': 'json',
  'solution_criteria': "The JSON must 

In [5]:
def run_prompt(test_case):
	"""Merges the prompt and test case input, then returns the result"""

	prompt = f"""
		Please solve the following task:
		{test_case["task"]}

		* Respond only with Python, Json, or a plain Regex
		* Do not add any comments or commentary or explanation
	"""

	messages = []
	add_user_message(messages, prompt)
	add_assistant_message(messages, "```code") # we could put ```python or ```regex or ```json, but ```code covers all 3
	output = chat(messages, stop_sequences=["```"])
	return output

In [6]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
		You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution against the solution criteria.

		Original Task:
		<task>
		{test_case["task"]}
		</task>

		Solution to Evaluate:
		<solution>
		{output}
		</solution>

        Solution Criteria:
        <solution-criteria>
        {test_case["solution_criteria"]}
		</solution-criteria>
        
		Output Format
		Provide your evaluation as a structured JSON object with the following fields, in this specific order:
		- "strengths": An array of 1-3 key strengths
		- "weaknesses": An array of 1-3 key areas for improvement
		- "reasoning": A concise explanation of your overall assessment
		- "score": A number between 1-10

		Respond with JSON. Keep your response concise and direct.
		Example response shape:
		{{
			"strengths": string[],
			"weaknesses": string[],
			"reasoning": string,
			"score": number
		}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)
  

In [7]:
import re
import ast

def validate_json(text):
	try:
		json.loads(text.strip())
		return 10
	except json.JSONDecodeError:
		return 0
	
def validate_python(text):
	try:
		ast.parse(text.strip())
		return 10
	except SyntaxError:
		return 0
	
def validate_regex(text):
	try:
		re.compile(text.strip())
		return 10
	except re.error:
		return 0
	
def grade_syntax(response, test_case):
	format = test_case["format"]
	if format == "json":
		return validate_json(response)
	elif format == "python":
		return validate_python(response)
	elif format == "regex":
		return validate_regex(response)
	else:
		return -1

In [8]:
def run_test_case(test_case):
	"""Calls run_prompt, then grades the result"""
	output = run_prompt(test_case)

	# model grading
	model_grade = grade_by_model(test_case, output)
	model_score = model_grade["score"]
	reasoning = model_grade["reasoning"]

	# code grading
	syntax_score = grade_syntax(output, test_case)

	score = (model_score + syntax_score) / 2

	return {
		"output" : output,
		"test_case" : test_case,
		"score" : score,
		"reasoning" : reasoning
	}


In [9]:
from statistics import mean

def run_eval(dataset):
	"""Loads the dataset and calls run_test_case with each case"""
	results = []

	for test_case in dataset:
		result = run_test_case(test_case)
		results.append(result)

	average_score = mean([result["score"] for result in results])
	print(f"Average score: {average_score}")
	return results

In [14]:
with open("dataset.json", "r") as file:
	dataset = json.load(file)

results = run_eval(dataset)

Average score: 8.666666666666666


In [15]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket_name(s3_uri):\n    match = re.match(r's3://([^/]+)', s3_uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a full S3 URI (e.g., s3://my-bucket-name/path/to/object) and extract just the bucket name",
      "format": "regex",
      "solution_criteria": "The regex must correctly extract the bucket name from S3 URIs in the format s3://bucket-name/... and handle bucket names with hyphens and numbers. It should return only the bucket name without the s3:// prefix or path."
    },
    "score": 8.5,
    "reasoning": "The solution correctly solves the core task of parsing and extracting bucket names from standard S3 URIs. The regex pattern is simple, efficient, and handles the specified requirements including hyphens and numbers. However, it lacks comprehensive input validation and AWS S3 bucket naming constraint validation, which would be important for production use